# DogBreedID — Task 04 / Nhiệm vụ 1
## Final Classification & Calibration Inference

Notebook này clone/pull repository `DGiang1126/dogbreedid-cv-project`, cài source Task 1, **mount Google Drive để lấy thư mục ảnh Stanford Dogs và checkpoint E4**, đọc artifact Temperature E6, sau đó chạy inference trên `data/splits/validation.csv`.

Luồng chạy: clone/pull repository → đưa source Task 1 vào runtime → kết nối Google Drive và tự tìm/cho phép chỉ định `Images`, checkpoint `best.pt`, artifact E6 → xác minh SHA-256 → unit tests synthetic → inference trên Validation → xuất NPZ raw logits + JSON metrics/metadata → sao chép output trở lại Google Drive → regression tests.

**Bảo vệ dữ liệu:** chỉ cho phép `validation`. Không đổi split sang Official Test, không fit lại Temperature E6, không chọn lại Threshold E7, không sửa frozen config, không push GitHub. Checkpoint trong Google Drive chỉ được chấp nhận nếu SHA-256 khớp đúng checkpoint đã được trưởng nhóm phê duyệt.


In [ ]:
# Google Colab: clone/pull DogBreedID repository
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/DGiang1126/dogbreedid-cv-project.git"
REPO_DIR = Path("/content/dogbreedid-cv-project")
BRANCH = "main"


def run_git(args, cwd=None, check=True):
    return subprocess.run(
        ["git", *args], cwd=str(cwd) if cwd else None,
        text=True, check=check, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    ).stdout.strip()

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(
            f"{REPO_DIR} đã tồn tại nhưng không phải Git repository. "
            "Đổi REPO_DIR hoặc xóa thư mục này trong Colab rồi chạy lại cell."
        )
    origin = run_git(["remote", "get-url", "origin"], cwd=REPO_DIR)
    normalized_origin = origin.removesuffix(".git").rstrip("/").lower()
    expected_origin = REPO_URL.removesuffix(".git").rstrip("/").lower()
    if normalized_origin != expected_origin:
        raise RuntimeError(
            f"Repository tại {REPO_DIR} có origin không đúng: {origin}\n"
            f"Origin mong đợi: {REPO_URL}. Hãy sửa REPO_DIR/REPO_URL trước khi chạy."
        )

    working_tree = run_git(["status", "--porcelain"], cwd=REPO_DIR)
    if working_tree:
        print("Repository đã tồn tại nhưng có thay đổi local; bỏ qua git pull để tránh ghi đè.")
        print("Nếu cần cập nhật main mới nhất, dùng runtime mới hoặc tự xử lý các thay đổi trước.")
    else:
        print(run_git(["fetch", "origin", BRANCH], cwd=REPO_DIR))
        current_branch = run_git(["branch", "--show-current"], cwd=REPO_DIR)
        if current_branch != BRANCH:
            print(run_git(["checkout", BRANCH], cwd=REPO_DIR))
        print(run_git(["pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR))
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(run_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)]))

REPO_ROOT = REPO_DIR.resolve()
print("\nRepository:", REPO_ROOT)
print("Git branch:", run_git(["branch", "--show-current"], cwd=REPO_ROOT))
print("Git SHA:", run_git(["rev-parse", "HEAD"], cwd=REPO_ROOT))


## 0B. Đưa các file Task 1 vào repository Colab

Repository được clone/pull ở cell trước. Vì các file Task 1 này chưa chắc đã có trên GitHub, cell sau sẽ yêu cầu bạn upload `Task1_Final_Inference_Package.zip` (chỉ cần làm khi package chưa được đặt tại `/content`). Cell sẽ chép các file inference/tests/runner từ package vào clone Colab và sao lưu file trùng tên trước khi ghi đè. Việc này chỉ thay đổi bản clone trong Colab, không push lên GitHub.


In [ ]:
# Upload package ZIP nếu chưa có, sau đó chép các file Task 1 vào bản clone Colab.
from pathlib import Path
import zipfile
import shutil
import subprocess
import sys
import importlib.util

BUNDLE_ZIP = Path("/content/Task1_Final_Inference_Package.zip")
if not BUNDLE_ZIP.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Không tìm thấy package ZIP trong /content. Cell này được thiết kế cho Google Colab; "
            "hãy đặt Task1_Final_Inference_Package.zip vào /content trước khi chạy."
        ) from exc
    print("Hãy upload file Task1_Final_Inference_Package.zip vừa tải về từ ChatGPT.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("Bạn chưa upload file ZIP. Hãy chạy lại cell và chọn Task1_Final_Inference_Package.zip.")
    preferred = [name for name in zip_names if Path(name).name == "Task1_Final_Inference_Package.zip"]
    chosen_name = preferred[0] if preferred else zip_names[0]
    uploaded_path = Path("/content") / chosen_name
    if uploaded_path.resolve() != BUNDLE_ZIP.resolve():
        shutil.copy2(uploaded_path, BUNDLE_ZIP)

# Chỉ lấy các file thuộc phạm vi Task 1, không giải nén tùy tiện toàn bộ archive vào repository.
allowed_files = [
    "src/inference/__init__.py",
    "src/inference/predictor.py",
    "scripts/run_final_inference.py",
    "tests/test_final_inference.py",
    "requirements-task1.txt",
    "README_TASK1.md",
    "docs/task_reports/thuan/20_10_task.md",
]
backup_root = Path("/content/task1_backup_before_overlay")
with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
    archive_names = set(zf.namelist())
    missing = [name for name in allowed_files if name not in archive_names]
    if missing:
        raise RuntimeError("Package ZIP thiếu file bắt buộc:\n" + "\n".join(missing))

    for rel in allowed_files:
        target = REPO_ROOT / rel
        if target.exists():
            backup = backup_root / rel
            backup.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, backup)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(zf.read(rel))
        print("Installed:", target.relative_to(REPO_ROOT))

# Cài các thư viện Task 1 còn thiếu; không cài lại/nâng cấp Torch trong Colab.
package_imports = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "yaml": "PyYAML",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "pytest": "pytest",
}
missing_packages = [pip_name for module_name, pip_name in package_imports.items()
                    if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print("Cài thư viện còn thiếu:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("Các thư viện Python cần thiết đã có sẵn.")

print("\nTask 1 files are ready in:", REPO_ROOT)
print("Backup của các file trùng tên (nếu có):", backup_root)
print("Không có lệnh git push nào được thực hiện.")


In [ ]:
# Imports và xác nhận môi trường sau khi clone repository + chuẩn bị source Task 1.
from pathlib import Path
import sys
import json
import subprocess
import platform
import numpy as np
import pandas as pd
import torch

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repository root:', REPO_ROOT)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 0C. Kết nối Google Drive để lấy dataset và checkpoint

Chạy cell tiếp theo để cấp quyền Google Drive. Tự động dò thư mục ảnh theo `relative_path` trong `validation.csv`, tìm checkpoint `best.pt` theo SHA-256 đã được phê duyệt và tìm artifact Temperature E6. Nếu biết chính xác đường dẫn, điền các biến cấu hình ở đầu cell trước khi chạy.

In [ ]:
# Mount Google Drive; lấy ảnh và checkpoint trực tiếp từ MyDrive.
from pathlib import Path
import os
import json
import hashlib
import pandas as pd

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "Cell này dành cho Google Colab. Hãy mở notebook bằng Colab để mount Google Drive."
    ) from exc

drive.mount('/content/drive', force_remount=False)
MYDRIVE = Path('/content/drive/MyDrive').resolve()
if not MYDRIVE.is_dir():
    raise FileNotFoundError(f"Google Drive chưa được mount đúng: {MYDRIVE}")

# Tùy chỉnh các đường dẫn này nếu muốn chỉ định thủ công.
# Ví dụ: DRIVE_IMAGES_DIR = Path('/content/drive/MyDrive/DogBreedID/Images')
#        DRIVE_CHECKPOINT_PATH = Path('/content/drive/MyDrive/DogBreedID/outputs/checkpoints/E4/seed_42/best.pt')
#        DRIVE_TEMPERATURE_ARTIFACT = Path('/content/drive/MyDrive/DogBreedID/outputs/metrics/E6/temperature.json')
DRIVE_SEARCH_ROOT = MYDRIVE
DRIVE_IMAGES_DIR = None
DRIVE_CHECKPOINT_PATH = None
DRIVE_TEMPERATURE_ARTIFACT = None
DRIVE_OUTPUT_ROOT = MYDRIVE / 'DogBreedID_Task04_Outputs'

from src.inference.predictor import (
    EXPECTED_CHECKPOINT_SHA256,
    DEFAULT_TEMPERATURE_ARTIFACT,
    sha256_file,
    _load_temperature,
)

VALIDATION_CSV_FOR_DISCOVERY = (REPO_ROOT / 'data' / 'splits' / 'validation.csv').resolve()
if not VALIDATION_CSV_FOR_DISCOVERY.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy Validation CSV: {VALIDATION_CSV_FOR_DISCOVERY}. "
        "Cần clone repository đúng trước khi dò thư mục ảnh."
    )
_validation_for_discovery = pd.read_csv(VALIDATION_CSV_FOR_DISCOVERY)
if 'relative_path' not in _validation_for_discovery.columns or _validation_for_discovery.empty:
    raise ValueError('validation.csv phải có cột relative_path và ít nhất một mẫu.')
_sample_rel_paths = [str(x).replace('\\', '/') for x in _validation_for_discovery['relative_path'].head(5)]


def _is_image_root(candidate: Path) -> bool:
    """Image root hợp lệ khi đọc được các đường dẫn mẫu trong Validation CSV."""
    try:
        candidate = candidate.expanduser().resolve()
        return candidate.is_dir() and all((candidate / Path(rel)).is_file() for rel in _sample_rel_paths)
    except (OSError, ValueError):
        return False


def _read_valid_e6(path: Path) -> bool:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
        return (
            isinstance(payload, dict)
            and payload.get('experiment') == 'E6'
            and payload.get('model_experiment') in (None, 'E4')
            and payload.get('checkpoint_seed') in (None, 42)
            and payload.get('split') in (None, 'calibration')
        )
    except Exception:
        return False

# 1) Tìm thư mục Images. Các đường dẫn quen thuộc được kiểm tra trước.
if DRIVE_IMAGES_DIR is not None:
    DRIVE_IMAGES_DIR = Path(DRIVE_IMAGES_DIR).expanduser().resolve()
    if not _is_image_root(DRIVE_IMAGES_DIR):
        raise FileNotFoundError(
            f"DRIVE_IMAGES_DIR không khớp các relative_path trong Validation CSV: {DRIVE_IMAGES_DIR}. "
            "Hãy trỏ tới thư mục trực tiếp chứa các folder giống chó."
        )
else:
    _image_candidates = [
        MYDRIVE / 'Images', MYDRIVE / 'StanfordDogs' / 'Images',
        MYDRIVE / 'Stanford Dogs' / 'Images', MYDRIVE / 'Dataset' / 'Images',
        MYDRIVE / 'dataset' / 'Images', MYDRIVE / 'datasets' / 'Images',
        MYDRIVE / 'datasets' / 'stanford_dogs' / 'Images',
        MYDRIVE / 'data' / 'raw' / 'Images',
        MYDRIVE / 'DogBreedID' / 'data' / 'raw' / 'Images',
        MYDRIVE / 'dogbreedid-cv-project' / 'data' / 'raw' / 'Images',
    ]
    DRIVE_IMAGES_DIR = next((p.resolve() for p in _image_candidates if _is_image_root(p)), None)

# 2) One bounded walk collects candidates. Set DRIVE_SEARCH_ROOT to a narrower folder
# if MyDrive contains a very large number of files.
_checkpoint_candidates = []
_temperature_candidates = []
_walking_images_candidates = []
if DRIVE_SEARCH_ROOT is not None and Path(DRIVE_SEARCH_ROOT).is_dir():
    _search_root = Path(DRIVE_SEARCH_ROOT).resolve()
    _skip_dirs = {'.git', '.trash', '.ipynb_checkpoints', 'node_modules', '__pycache__'}
    _max_visited_dirs = 12000
    _max_depth = 10
    _visited_dirs = 0
    for _current, _dirs, _files in os.walk(_search_root, topdown=True):
        _current_path = Path(_current)
        try:
            _depth = len(_current_path.relative_to(_search_root).parts)
        except ValueError:
            _dirs[:] = []
            continue
        _dirs[:] = [d for d in _dirs if d.lower() not in _skip_dirs and not d.startswith('.')]
        if _depth >= _max_depth:
            _dirs[:] = []
        _visited_dirs += 1
        if _visited_dirs > _max_visited_dirs:
            print(f"Đã chạm giới hạn dò {_max_visited_dirs} thư mục; nếu chưa tìm thấy file, đặt DRIVE_SEARCH_ROOT hẹp hơn.")
            break

        _lower_files = {f.lower(): f for f in _files}
        for _name in ('best.pt', 'best.pth'):
            if _name in _lower_files:
                _checkpoint_candidates.append((_current_path / _lower_files[_name]).resolve())
        for _filename in _files:
            _low = _filename.lower()
            _full_low = str(_current_path / _filename).lower()
            if _low.endswith('.json') and ('temperature' in _full_low or 'calibration' in _full_low or '/e6/' in _full_low or _full_low.endswith('/e6.json')):
                _temperature_candidates.append((_current_path / _filename).resolve())

        if DRIVE_IMAGES_DIR is None and _current_path.name.lower() == 'images' and _is_image_root(_current_path):
            _walking_images_candidates.append(_current_path.resolve())
            # Avoid descending into the large image tree once the root is identified.
            _dirs[:] = []

if DRIVE_IMAGES_DIR is None:
    _unique_images = list(dict.fromkeys(_walking_images_candidates))
    if len(_unique_images) == 1:
        DRIVE_IMAGES_DIR = _unique_images[0]
    elif len(_unique_images) > 1:
        print('Tìm thấy nhiều thư mục Images hợp lệ:')
        for _p in _unique_images:
            print(' -', _p)
        raise RuntimeError('Có nhiều thư mục Images khớp. Hãy gán DRIVE_IMAGES_DIR bằng đường dẫn chính xác rồi chạy lại cell.')
    else:
        raise FileNotFoundError(
            "Không tự tìm thấy thư mục ảnh trong Google Drive. Hãy đặt DRIVE_IMAGES_DIR = Path('/content/drive/MyDrive/.../Images') "
            'ở phần cấu hình đầu cell và chạy lại. Thư mục phải trực tiếp chứa các folder giống chó.'
        )

# 3) Chọn checkpoint E4 đúng hash. Mặc định ưu tiên checkpoint trong Drive và không bỏ qua mismatch.
if DRIVE_CHECKPOINT_PATH is not None:
    DRIVE_CHECKPOINT_PATH = Path(DRIVE_CHECKPOINT_PATH).expanduser().resolve()
    if not DRIVE_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f'Không thấy checkpoint: {DRIVE_CHECKPOINT_PATH}')
    _actual_hash = sha256_file(DRIVE_CHECKPOINT_PATH)
    if _actual_hash.lower() != EXPECTED_CHECKPOINT_SHA256.lower():
        raise RuntimeError(
            'Checkpoint trong DRIVE_CHECKPOINT_PATH không đúng SHA-256 đã phê duyệt. '
            f'Expected={EXPECTED_CHECKPOINT_SHA256}; actual={_actual_hash}; path={DRIVE_CHECKPOINT_PATH}'
        )
else:
    _unique_ckpts = list(dict.fromkeys(_checkpoint_candidates))
    _unique_ckpts.sort(key=lambda p: (0 if '/outputs/checkpoints/e4/seed_42/best.pt' in p.as_posix().lower() else 1, len(p.parts)))
    DRIVE_CHECKPOINT_PATH = None
    _checkpoint_hash_report = []
    for _candidate in _unique_ckpts:
        try:
            _actual_hash = sha256_file(_candidate)
            _checkpoint_hash_report.append((str(_candidate), _actual_hash))
            if _actual_hash.lower() == EXPECTED_CHECKPOINT_SHA256.lower():
                DRIVE_CHECKPOINT_PATH = _candidate
                break
        except Exception as _exc:
            _checkpoint_hash_report.append((str(_candidate), f'ERROR: {_exc}'))
    if DRIVE_CHECKPOINT_PATH is None:
        _details = '\n'.join(f' - {p}: {h}' for p, h in _checkpoint_hash_report[:12]) or ' - Không tìm thấy best.pt/best.pth trong phạm vi đã dò.'
        raise FileNotFoundError(
            'Không tìm thấy checkpoint E4 đúng hash trong Google Drive. Không được bỏ qua hash check. '
            "Đặt DRIVE_CHECKPOINT_PATH = Path('/content/drive/MyDrive/.../best.pt') ở đầu cell để chọn thủ công.\n"
            f'Expected SHA-256: {EXPECTED_CHECKPOINT_SHA256}\nCác candidate đã kiểm tra:\n{_details}'
        )

# 4) Temperature: dùng artifact trong repo nếu có và hợp lệ; nếu không, dùng artifact E6 trong Drive.
_repo_temp = (REPO_ROOT / DEFAULT_TEMPERATURE_ARTIFACT).resolve()
if DRIVE_TEMPERATURE_ARTIFACT is not None:
    DRIVE_TEMPERATURE_ARTIFACT = Path(DRIVE_TEMPERATURE_ARTIFACT).expanduser().resolve()
    if not DRIVE_TEMPERATURE_ARTIFACT.is_file() or not _read_valid_e6(DRIVE_TEMPERATURE_ARTIFACT):
        raise FileNotFoundError(f'Đường dẫn không phải artifact E6 hợp lệ: {DRIVE_TEMPERATURE_ARTIFACT}')
else:
    if _repo_temp.is_file() and _read_valid_e6(_repo_temp):
        DRIVE_TEMPERATURE_ARTIFACT = _repo_temp
    else:
        _temp_matches = [p for p in dict.fromkeys(_temperature_candidates) if _read_valid_e6(p)]
        _temp_matches.sort(key=lambda p: (0 if p.name.lower() == 'temperature.json' else 1, len(p.parts)))
        if len(_temp_matches) == 1:
            DRIVE_TEMPERATURE_ARTIFACT = _temp_matches[0]
        elif len(_temp_matches) > 1:
            print('Tìm thấy nhiều artifact E6 hợp lệ:')
            for _p in _temp_matches:
                print(' -', _p)
            raise RuntimeError('Có nhiều artifact E6 hợp lệ. Hãy gán DRIVE_TEMPERATURE_ARTIFACT bằng artifact đã nghiệm thu rồi chạy lại cell.')
        else:
            raise FileNotFoundError(
                'Không tìm thấy JSON artifact Temperature E6 hợp lệ. Hãy đặt DRIVE_TEMPERATURE_ARTIFACT = '
                "Path('/content/drive/MyDrive/.../temperature.json') ở đầu cell. Không nhập tay Temperature thay cho artifact."
            )

# Final validation through the project's E6 loader.
_temperature_value, _temperature_payload = _load_temperature(DRIVE_TEMPERATURE_ARTIFACT)
print('Google Drive mounted:', MYDRIVE)
print('Images directory:', DRIVE_IMAGES_DIR)
print('E4 checkpoint:', DRIVE_CHECKPOINT_PATH)
print('Checkpoint SHA-256: VERIFIED')
print('E6 Temperature artifact:', DRIVE_TEMPERATURE_ARTIFACT)
print('Temperature loaded from artifact:', repr(_temperature_value))
print('Drive output root:', DRIVE_OUTPUT_ROOT)


## 1. Cấu hình chạy

Cell Google Drive ở phía trên sẽ mount `MyDrive`, thử tự tìm thư mục `Images` có các file trùng `relative_path` của Validation, checkpoint `best.pt` có SHA-256 đúng, và artifact JSON E6. Nếu có nhiều thư mục / bố cục khác thường, bạn có thể đặt đường dẫn chính xác ở cell Google Drive trong các biến `DRIVE_IMAGES_DIR`, `DRIVE_CHECKPOINT_PATH`, `DRIVE_TEMPERATURE_ARTIFACT` trước khi chạy cell đó.

Không cần tải bộ ảnh hoặc checkpoint lên `/content`; notebook dùng trực tiếp các đường dẫn trong Google Drive. Runner tiếp tục ghi output tạm trong repository Colab để tuân thủ Data Contract, sau đó có cell sao chép các file kết quả về `MyDrive/DogBreedID_Task04_Outputs/`.


In [ ]:
# Task 04 chỉ cho phép Validation. Ảnh/checkpoint/artifact E6 đã được resolve từ Google Drive cell bên trên.
from pathlib import Path
import torch

SPLIT_NAME = 'validation'  # Không đổi thành test/official_test
SPLIT_CSV = REPO_ROOT / 'data' / 'splits' / 'validation.csv'
CHECKPOINT_PATH = Path(DRIVE_CHECKPOINT_PATH).resolve()
TEMPERATURE_ARTIFACT = Path(DRIVE_TEMPERATURE_ARTIFACT).resolve()
IMAGES_DIR = Path(DRIVE_IMAGES_DIR).resolve()
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'predictions' / 'task04_validation'

BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

required_files = [
    REPO_ROOT / 'configs' / 'data.yaml',
    REPO_ROOT / 'configs' / 'experiments' / 'e4_cutmix.yaml',
    SPLIT_CSV,
    CHECKPOINT_PATH,
    TEMPERATURE_ARTIFACT,
    IMAGES_DIR,
    REPO_ROOT / 'src' / 'models' / 'model_factory.py',
    REPO_ROOT / 'src' / 'data' / 'dataset.py',
    REPO_ROOT / 'src' / 'data' / 'transforms.py',
    REPO_ROOT / 'src' / 'evaluation' / 'metrics.py',
    REPO_ROOT / 'src' / 'calibration' / 'calibration_metrics.py',
    REPO_ROOT / 'src' / 'calibration' / 'temperature_scaling.py',
    REPO_ROOT / 'data' / 'metadata' / 'class_mapping.json',
]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError('Thiếu file/thư mục cần thiết:\n' + '\n'.join(missing_files))

print('Split CSV:', SPLIT_CSV)
print('Images from Google Drive:', IMAGES_DIR)
print('Checkpoint from Google Drive:', CHECKPOINT_PATH)
print('Temperature artifact:', TEMPERATURE_ARTIFACT)
print('Output dir (temporary in Colab repo):', OUTPUT_DIR)
print('Device:', DEVICE)


## 2. Xác minh checkpoint, E6 artifact và split

Checkpoint phải trùng SHA-256 được trưởng nhóm phê duyệt. Sai hash sẽ dừng, không có tùy chọn bỏ qua. Temperature phải được đọc từ JSON E6, không nhập tay hoặc refit.


In [ ]:
from src.inference.predictor import (
    EXPECTED_CHECKPOINT_SHA256,
    assert_task04_split_allowed,
    sha256_file,
    verify_checkpoint_hash,
    _load_temperature,
)

assert_task04_split_allowed(SPLIT_NAME)
actual_checkpoint_sha = verify_checkpoint_hash(CHECKPOINT_PATH)
assert actual_checkpoint_sha == EXPECTED_CHECKPOINT_SHA256

temperature, temperature_artifact_payload = _load_temperature(TEMPERATURE_ARTIFACT)
assert temperature_artifact_payload.get('experiment') == 'E6'
assert temperature_artifact_payload.get('model_experiment') in (None, 'E4')

print('Checkpoint SHA-256:', actual_checkpoint_sha)
print('Checkpoint hash verified: PASS')
print('E6 Temperature (loaded from artifact):', repr(temperature))
print('E6 artifact split:', temperature_artifact_payload.get('split'))
print('Approved split:', SPLIT_NAME)


## 3. Tìm thư mục ảnh và kiểm tra shared dataset/preprocessing

Dataset dùng đúng `DogBreedDataset` và `build_transforms(..., split='validation')` của repository. Cell này chỉ đọc một mẫu để kiểm tra input tensor; không train và chưa thực hiện evaluation.


In [ ]:
from src.data.dataset import DogBreedDataset
from src.data.transforms import build_transforms
import yaml

# IMAGES_DIR phải là đường dẫn đã mount trên Google Drive, được kiểm tra ở cell Drive.
IMAGES_DIR = Path(IMAGES_DIR).expanduser().resolve()
if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy thư mục ảnh trên Google Drive: {IMAGES_DIR}')
_csv_rel_paths = [str(x).replace('\\', '/') for x in pd.read_csv(SPLIT_CSV)['relative_path'].head(5)]
if not all((IMAGES_DIR / Path(rel)).is_file() for rel in _csv_rel_paths):
    raise FileNotFoundError(
        'DRIVE_IMAGES_DIR không chứa các relative_path trong Validation CSV. '
        'Hãy trỏ tới đúng thư mục trực tiếp chứa các folder giống chó (thường là Images).'
    )

with (REPO_ROOT / 'configs' / 'data.yaml').open('r', encoding='utf-8') as f:
    data_payload = yaml.safe_load(f)
data_config = data_payload.get('data', data_payload)
validation_transform = build_transforms(data_payload, split='validation')
validation_dataset = DogBreedDataset(
    csv_path=SPLIT_CSV,
    images_dir=IMAGES_DIR,
    transform=validation_transform,
)

sample = validation_dataset[0]
print('Images root (Google Drive):', IMAGES_DIR)
print('Validation samples:', len(validation_dataset))
print('Sample image_id:', sample['relative_path'])
print('Sample label:', sample['label'])
print('Sample image tensor shape:', tuple(sample['image'].shape))
assert tuple(sample['image'].shape) == (3, 224, 224)
assert SPLIT_CSV.name == 'validation.csv'


## 4. Chạy unit tests synthetic

Các tests kiểm tra cấu trúc NPZ, checkpoint hash guard, từ chối Official Test, Temperature Scaling không đổi Top-1, đầu vào sai shape, tính tái lập khi đổi batch size và việc weights không thay đổi. Dữ liệu của các unit tests là synthetic, không phải Official Test.


In [ ]:
test_result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_final_inference.py', '-q'],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
if test_result.returncode != 0:
    raise RuntimeError(f'test_final_inference.py FAILED (exit code {test_result.returncode})')
print('Targeted Task 1 tests: PASS')


## 5. Chạy toàn bộ Final Inference trên Validation

Runner sẽ nạp model bằng model builder E4 hiện có, xác minh checkpoint SHA-256, dùng `torch.load(..., weights_only=True)` cùng `load_state_dict(..., strict=True)`, gọi `model.eval()` và `torch.inference_mode()`, rồi lưu raw logits. Không có bước training/optimizer.


In [ ]:
from src.inference.predictor import run_final_inference

result = run_final_inference(
    repo_root=REPO_ROOT,
    split_name=SPLIT_NAME,
    split_csv=SPLIT_CSV,
    images_dir=IMAGES_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    temperature_artifact=TEMPERATURE_ARTIFACT,
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    device=DEVICE,
)

print('N samples:', result['metrics']['n_samples'])
print('Checkpoint SHA-256:', result['checkpoint_sha256'])
print('Device:', result['metadata']['runtime']['device'])
print('Outputs:')
for key, path in result['paths'].items():
    print(f'  {key}: {path}')


## 6. Đọc Classification Metrics và Calibration Metrics

Các metrics phân loại tái sử dụng `src.evaluation.metrics.classification_metrics`. NLL/ECE/Brier tái sử dụng hàm đã nghiệm thu ở E6, với 15 bins và Temperature lấy từ artifact. Brier Score giữ nguyên định nghĩa multiclass hiện tại của project.


In [ ]:
from IPython.display import display, Image

classification = result['metrics']['classification']
calibration = result['metrics']['calibration']

summary = pd.DataFrame([
    {'metric': 'Top-1 Accuracy', 'value': classification['top1_accuracy']},
    {'metric': 'Top-5 Accuracy', 'value': classification['top5_accuracy']},
    {'metric': 'Macro-F1', 'value': classification['macro_f1']},
    {'metric': 'NLL before', 'value': calibration['before_temperature_scaling']['nll']},
    {'metric': 'NLL after', 'value': calibration['after_temperature_scaling']['nll']},
    {'metric': 'ECE before (15 bins)', 'value': calibration['before_temperature_scaling']['ece']},
    {'metric': 'ECE after (15 bins)', 'value': calibration['after_temperature_scaling']['ece']},
    {'metric': 'Brier before', 'value': calibration['before_temperature_scaling']['multiclass_brier']},
    {'metric': 'Brier after', 'value': calibration['after_temperature_scaling']['multiclass_brier']},
])
display(summary)
if result['paths']['reliability_diagram'] and Path(result['paths']['reliability_diagram']).is_file():
    display(Image(filename=result['paths']['reliability_diagram']))
print('Top-1 unchanged after Temperature Scaling:', calibration['top1_prediction_unchanged'])
assert calibration['top1_prediction_unchanged'] is True

display(pd.DataFrame(classification['per_class']))


## 7. Kiểm tra NPZ và metadata sau khi chạy

Kiểm tra lại file đã lưu bằng `allow_pickle=False`, đúng ba trường Data Contract, shape/dtype và khớp image IDs/labels với Validation CSV.


In [ ]:
from src.inference.predictor import load_raw_predictions

npz_path = Path(result['paths']['raw_predictions_npz'])
loaded = load_raw_predictions(npz_path)
assert set(loaded.keys()) == {'logits', 'labels', 'image_ids'}
assert loaded['logits'].shape == (len(validation_dataset), 30)
assert loaded['labels'].shape == (len(validation_dataset),)
assert loaded['image_ids'].shape == (len(validation_dataset),)
assert loaded['labels'].dtype == np.int64
assert loaded['image_ids'].dtype.kind == 'U'
assert np.isfinite(loaded['logits']).all()
assert len(set(loaded['image_ids'].tolist())) == len(loaded['image_ids'])

csv_frame = pd.read_csv(SPLIT_CSV)
assert loaded['image_ids'].tolist() == csv_frame['relative_path'].astype(str).tolist()
assert np.array_equal(loaded['labels'], csv_frame['class_id'].astype(np.int64).to_numpy())

with Path(result['paths']['metadata_json']).open('r', encoding='utf-8') as f:
    saved_metadata = json.load(f)
assert saved_metadata['split'] == 'validation'
assert saved_metadata['safety']['official_test_used'] is False
assert saved_metadata['checkpoint_sha256'] == EXPECTED_CHECKPOINT_SHA256

print('NPZ Data Contract: PASS')
print('Metadata/split/checkpoint checks: PASS')
print('Output directory:', result['outputs_dir'])


## 8B. Sao chép kết quả về Google Drive để lưu bền vững

Kết quả được tạo trong `/content/dogbreedid-cv-project/outputs/...` trước, sau đó sao chép bản kết quả đã kiểm tra sang `MyDrive/DogBreedID_Task04_Outputs/`. Mỗi lần chạy dùng một thư mục riêng để tránh ghi đè kết quả cũ.

In [ ]:
from datetime import datetime
import shutil

_drive_run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_RUN_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / f'task04_validation_{_drive_run_stamp}'
DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
_copied_outputs = []
for _key, _source in result['paths'].items():
    if not _source:
        continue
    _source_path = Path(_source)
    if _source_path.is_file():
        _target_path = DRIVE_RUN_OUTPUT_DIR / _source_path.name
        shutil.copy2(_source_path, _target_path)
        _copied_outputs.append((_key, _target_path))

print('Đã sao chép kết quả sang Google Drive:')
for _key, _path in _copied_outputs:
    print(f' - {_key}: {_path}')
if not _copied_outputs:
    raise RuntimeError("Không có output nào được sao chép về Google Drive; hãy kiểm tra result['paths'].")


## 9. Trước khi bàn giao

Sau khi notebook chạy thành công, cần lưu lại kết quả `pytest` và các đường dẫn output trong `docs/task_reports/thuan/20_10_task.md`. Báo cáo không được đánh dấu PASS nếu chưa có kết quả thật.

Output tạm nằm trong `outputs/predictions/task04_validation/`; bản sao bền vững cũng nằm trong Google Drive tại thư mục được in ra ở cell sao chép kết quả:
- `validation_raw_predictions.npz`
- `validation_metrics.json`
- `validation_metadata.json`
- `validation_reliability_diagram.png` (nếu Matplotlib khả dụng)

Cuối cùng chạy `python -m pytest -q` trong repository để kiểm tra regression toàn dự án. Không push/merge trong notebook; việc đó thực hiện sau theo quy trình nhóm.


In [ ]:
# Optional final regression check. This can take longer than the targeted tests.
RUN_FULL_REGRESSION = True  # Required before hand-off: run the complete repository test suite.
if RUN_FULL_REGRESSION:
    full_result = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q'],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )
    print(full_result.stdout)
    if full_result.stderr:
        print(full_result.stderr)
    if full_result.returncode != 0:
        raise RuntimeError(f'Full repository regression tests FAILED (exit code {full_result.returncode})')
    print('Full repository regression tests: PASS')
else:
    print('Full regression chưa chạy. Đặt RUN_FULL_REGRESSION=True và chạy cell này trước khi bàn giao.')
